<a href="https://colab.research.google.com/github/joshuajhchoi/ai2learn/blob/master/Random_Forest_from_Scratch_kr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

####1단계: 필요 라이브러리 임포트 및 데이터 준비

In [ ]:
import numpy as np
from collections import Counter
from sklearn.datasets import make_classification # 임의의 데이터 생성을 위한 라이브러리 (필요시)

####2단계: 결정 트리(Decision Tree) 클래스 정의

In [ ]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.tree = None

    def _entropy(self, y):
        # 불순도 측정 함수 (엔트로피 사용)
        counts = np.bincount(y)
        probabilities = counts[np.nonzero(counts)] / len(y)
        return -np.sum(probabilities * np.log2(probabilities))

    def _information_gain(self, X, y, split_idx):
        # 정보 획득량 계산 함수
        parent_entropy = self._entropy(y)
        left_indices, right_indices = self._split_data(X, split_idx)
        n = len(y)
        n_left, n_right = len(left_indices), len(right_indices)
        if n_left == 0 or n_right == 0:
            return 0
        weighted_child_entropy = (n_left / n) * self._entropy(y[left_indices]) + \
                                 (n_right / n) * self._entropy(y[right_indices])
        return parent_entropy - weighted_child_entropy

    def _split_data(self, X, split_idx):
        # 데이터를 두 그룹으로 나누는 함수
        split_value = X[split_idx[0], split_idx[1]]
        left_indices = np.where(X[:, split_idx[1]] <= split_value)[0]
        right_indices = np.where(X[:, split_idx[1]] > split_value)[0]
        return left_indices, right_indices

    def _find_best_split(self, X, y):
        # 최적의 분할 기준을 찾는 함수
        best_gain = -1
        best_split = None
        for i in range(X.shape[0]): # 모든 데이터 포인트에 대해
            for j in range(X.shape[1]): # 모든 feature에 대해
                split_idx = (i, j)
                gain = self._information_gain(X, y, split_idx)
                if gain > best_gain:
                    best_gain = gain
                    best_split = split_idx
        return best_split

    def _build_tree(self, X, y, depth=0):
        # 결정 트리 생성 함수 (재귀 호출)
        if len(y) < self.min_samples_split or (self.max_depth is not None and depth >= self.max_depth) or len(set(y)) == 1:
            return Counter(y).most_common(1)[0][0] # leaf node
        best_split = self._find_best_split(X, y)
        if best_split is None: # 더 이상 분할할 feature가 없는 경우
            return Counter(y).most_common(1)[0][0] # leaf node
        left_indices, right_indices = self._split_data(X, best_split)
        left_X, left_y = X[left_indices], y[left_indices]
        right_X, right_y = X[right_indices], y[right_indices]
        left_subtree = self._build_tree(left_X, left_y, depth + 1)
        right_subtree = self._build_tree(right_X, right_y, depth + 1)
        return {'split': best_split, 'left': left_subtree, 'right': right_subtree}

    def fit(self, X, y):
        # 결정 트리 학습 함수
        self.tree = self._build_tree(X, y)

    def _predict_one(self, x):
        # 하나의 데이터 포인트에 대한 예측 함수 (재귀 호출)
        subtree = self.tree
        while isinstance(subtree, dict):
            split_idx = subtree['split']
            split_value = X[split_idx[0], split_idx[1]]
            if x[split_idx[1]] <= split_value:
                subtree = subtree['left']
            else:
                subtree = subtree['right']
        return subtree

    def predict(self, X):
        # 여러 데이터 포인트에 대한 예측 함수
        return np.array([self._predict_one(x) for x in X])

####3단계: 랜덤 포레스트(Random Forest) 클래스 정의

In [ ]:
class RandomForest:
    def __init__(self, n_estimators=100, min_samples_split=2, max_depth=None, max_features=None):
        self.n_estimators = n_estimators
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.max_features = max_features
        self.forest = []

    def fit(self, X, y):
        # 랜덤 포레스트 학습 함수
        n_samples = X.shape[0]
        for _ in range(self.n_estimators):
            # 1. Bagging: 데이터 중복 추출
            sample_indices = np.random.choice(n_samples, n_samples, replace=True)
            X_sample, y_sample = X[sample_indices], y[sample_indices]

            # 2. Feature Randomness: feature 임의 선택
            if self.max_features is None:
                n_features = X.shape[1]
            else:
                n_features = int(self.max_features * X.shape[1])
            feature_indices = np.random.choice(X.shape[1], n_features, replace=False)
            X_sample = X_sample[:, feature_indices]

            # 3. 결정 트리 학습
            tree = DecisionTree(min_samples_split=self.min_samples_split, max_depth=self.max_depth)
            tree.fit(X_sample, y_sample)
            self.forest.append((tree, feature_indices))

    def predict(self, X):
        # 랜덤 포레스트 예측 함수
        predictions = []
        for tree, feature_indices in self.forest:
            X_subset = X[:, feature_indices]
            predictions.append(tree.predict(X_subset))
        # 앙상블: 다수결 투표
        final_predictions = np.array([Counter(preds).most_common(1)[0][0] for preds in zip(*predictions)])
        return final_predictions

####4단계: 데이터 생성 (예시)

In [ ]:
X, y = make_classification(n_samples=100, n_features=5, n_informative=3, n_redundant=1, random_state=42)

####5단계: 랜덤 포레스트 모델 생성 및 학습

In [ ]:
rf_model = RandomForest(n_estimators=50, max_depth=4)
rf_model.fit(X, y)

####6단계: 예측 수행

In [ ]:
predictions = rf_model.predict(X)

####7단계: 결과 확인

In [ ]:
print(predictions)

[0 1 1 1 0 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 1 0 1 0 1 0 0 1 1 1 1 1 1 1 0 1 1
 1 1 1 0 0 1 0 1 1 1 1 0 0 0 0 1 1 0 0 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 0 1 1
 0 0 1 1 0 0 1 0 1 1 0 1 1 1 1 0 0 1 0 0 0 0 0 0 1 0]


####8단계: 모델 평가 (예시)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y, predictions)
print(f"Accuracy: {accuracy}")

Accuracy: 0.9
